# Basic preparations: MERIT-Hydro-Geofabric for Canada and transboundary river basins

In this Notebook, the geospatial fabric for the "Canada and transboundary river basin" is extracted from the `MERIT-Hydro-Basins` dataset.

Access to raw `MERIT-Basins` dataset is needed. Fir HPC, it is located in the following path:

```console
/project/6102189/data/misc-data/MERIT-Basins
```

The version used is `MERIT_Hydro_v07_Basins_v01_bugfix1` which is a directory under the root directory of the dataset.

Let's get started with our workflow and import necessary Python libraries:

In [1]:
# import geopandas as gpd # version 0.14.0
import pandas as pd # version 1.4.0
import numpy as np # version 1.22.2
import matplotlib.pyplot as plt # version 3.5.1
import geopandas as gpd # version 0.14.3

from shapely.geometry import Point # version 2.0.1

import hydrant.topology.geom as gm # version 0.1.0-dev1

import subprocess # built-in Python 3.10.2
import os # built-in Python 3.10.2
import glob # built-in Python 3.10.2

`Hydrant` is important in this Notebook. We are only using the `topology.geom`etry module.

Path definitions (system dependant - modify accordingly):

In [2]:
# paths to geofabric data: merit-basins provided in `rrg-alpie`
merit_basins_root_path = '/project/6102189/data/misc-data/MERIT-Basins/MERIT_Hydro_v07_Basins_v01_bugfix1'
merit_basins_geom_path = os.path.join(merit_basins_root_path, 'pfaf_level_02')
merit_basins_nca_path = os.path.join(merit_basins_root_path, 'coastal_hillslopes')

# output paths
output_path = './CanTrans-merit-geofabric/'

# Reading `MERIT-Basins` Geospatial Fabric Dataset

## `MERIT-Basins` Geospatial Layers

Upon **visual** inspection (you may use `QGIS` or similar programs), to identify the domain of interest:

As you may see in the cell below, we are using Python `list`s to enable reading multiple layers at once. There are cases where a basin of interest is shared between multiple `pfaf` layers.

For now, let's read the files, one by one:

In [ ]:
# Merging multiple catchments (subbasin) vector layers to one layer for the Canada-Transboundary basins
# read the file names of all layes required for the study domain
cat_files = ['cat_pfaf_71_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',           
             'cat_pfaf_72_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'cat_pfaf_73_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'cat_pfaf_74_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'cat_pfaf_78_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'cat_pfaf_81_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'cat_pfaf_82_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'cat_pfaf_83_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'cat_pfaf_84_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'cat_pfaf_85_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'cat_pfaf_86_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
            ]
# rivers (river segments)
riv_files = ['riv_pfaf_71_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',             
             'riv_pfaf_72_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'riv_pfaf_73_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',             
             'riv_pfaf_74_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'riv_pfaf_78_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'riv_pfaf_81_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'riv_pfaf_82_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'riv_pfaf_83_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'riv_pfaf_84_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'riv_pfaf_85_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
             'riv_pfaf_86_MERIT_Hydro_v07_Basins_v01_bugfix1.shp',
            ]
# non-contributing catchments (those without any river segments defined for them)
nca_files = ['hillslope_71_clean.shp',
             'hillslope_72_clean.shp',
             'hillslope_73_clean.shp',
             'hillslope_74_clean.shp',
             'hillslope_78_clean.shp',
             'hillslope_81_clean.shp',
             'hillslope_82_clean.shp',
             'hillslope_83_clean.shp',
             'hillslope_84_clean.shp',
             'hillslope_85_clean.shp',
             'hillslope_86_clean.shp',
            ]
# reading in data in an iterative manner
cat = pd.concat([gpd.read_file(os.path.join(merit_basins_geom_path, f)) for f in cat_files])
riv = pd.concat([gpd.read_file(os.path.join(merit_basins_geom_path, f)) for f in riv_files])
nca = pd.concat([gpd.read_file(os.path.join(merit_basins_nca_path, f)) for f in nca_files])

Since `MERIT-Basins` layers do not come with correct Coordinate Reference System (CRS) information, we need to specify this manually. The `EPSG` code for the `MERIT-Basins` layer is `4326`. Please refer to the following for more information: https://en.wikipedia.org/wiki/EPSG_Geodetic_Parameter_Dataset

In [ ]:
# specifying epsg:4326 for all the MERIT-Basins layers
cat.set_crs(epsg=4326, inplace=True)
nca.set_crs(epsg=4326, inplace=True)
riv.set_crs(epsg=4326, inplace=True)

# Show the EPSG of all geospatial layers
print(f'`cat` CRS: {cat.crs}')
print(f'`riv` CRS: {riv.crs}')
print(f'`nca` CRS: {nca.crs}')

Based on the information above, we use `hydrant` to extract all the river segment of sub-basins upstream of the coordinates above.

# Preparing `cat`, `riv`, and `nca` objects for `Canada and transboundary river basins domain`

## Preparing `MERIT-Basins` Layers

Before subsetting the entire layer #71-86 of the `MERIT-Basins` dataset, we have to assure the layers are ready to be further processed by the `Hydrant` package. Fortunately, `Hydrant` provides necessary functionalities to work with this specific geospatial fabric (applicable to any geospatial fabric in reality). 

In doing so, Hydrant's `geom` module provides the `prepare_cat(...)` function to prepare the `MERIT-Basins` geosptial fabric's sub-basins (or catchments) for the next post-processing steps. Please note that since the non-contributing areas (`nca`) are technically considered sub-basins, they are taken care of using this functionality of `Hydrant`:

In [ ]:
# Hydrant's `geom` module provides the `prepare_cat`
# function to prepare the `MERIT-Basins` geosptial
# fabric for next post-processing steps

catchments = gm.prepare_cat(
    cat=cat, # 
    cat_col_id='COMID',
    cst=nca,
    cst_col_mapper={'FID':'COMID'},
    cst_col_id='COMID'
)

# You may see the "docstring" of the `gm.prepare_cat`
# function by running:
# >>> gm.prepare_cat?
# in a separate Jupyter cell (without the >>>), or by
# running simply:
# >>> print(gm.prepare_cat.__doc__)

Similarly, the `geom` module provides the `prepare_riv(...)` function to prepare the `MERIT-Basins` geospatial fabric's river segments for the next post-processing steps:

In [ ]:
# Similarly, the `geom` module provides the
# `prepare_riv` function to prepare the `MERIT-Basins`
# geospatial fabric's river segments for the next
# post-processing steps:

rivers = gm.prepare_riv(
    riv=riv,
    riv_cols={
        'id':'COMID',
        'next_id':'NextDownID',
        'slope':'slope',
        'length':'lengthkm',
        'length_direct':'lengthdir'
    },
    cat=catchments,
    cat_cols={
        'id':'COMID',
        'hillslope':'hillslope',
        'geom':'geometry'
    }
)

# You may see the "docstring" of the `gm.prepare_riv`
# function by running:
# >>> gm.prepare_riv?
# in a separate Jupyter cell (without the >>>), or by
# running simply:
# >>> print(gm.prepare_riv.__doc__)

In Python, you may always access the "docstring" documentations for the functions and classes by running:

```python
>>> print(func.__doc__)
```

Or, if you are working in the Jupyter environment, you may simply run the following in a separate Jupyter cell:
```ipython
[ln1] func?
```

If you are interested in reading up on the functionality of each function used above, use the mentioned methods to print the "docstrings".

Therefore, if you would like to read up on the `gm.prepare_cat(...)` or `gm.prepare_riv(...)` functionality, simply uncomment and execute the following cell:

In [1]:
# gm.prepare_cat?

In [2]:
# gm.prepare_riv?

## Subsetting Sub-basins and River Segments Upstream of the Canada and transboundary riverbasin domain

First, we need to find the `MERIT-Basins`'s sub-basin where this gauge is located. We use `geopandas` capability to do a quick intersection between the two layers (`MERIT-Basins` sub-basins (Multi-)Polygons and all point of interest (outlets)).

In [ ]:
# Read the study domain shapefile and prepare the Outlets
CanTrans = gpd.read_file('./CanTrans-merit-geofabric/CanTrans_MERIT_StudyDomain.shp')
CanTrans = CanTrans.rename(columns={'COMID': 'ID'})

# Ensure CRS match
if catchments.crs is not None:
    # Reproject CanTrans to cat's CRS
    CanTrans = CanTrans.to_crs(catchments.crs)
    
# Confirm the new CRS
# print(f'CanTrans CRS after projection: {CanTrans.crs}')

# Compute interior centroids
basins = catchments.copy()
basins['interior_centroid'] = basins.geometry.apply(lambda geom: geom.representative_point())
# Create GeoDataFrame of centroids with COMID
centroid_gdf = gpd.GeoDataFrame(
    basins[['COMID']].copy(),
    geometry=basins['interior_centroid'],
    crs=basins.crs
)
# Spatial join: keep only centroids within CanTrans polygons
centroids_within_target = gpd.sjoin(centroid_gdf, CanTrans, predicate='within', how='inner')
# Extract unique COMIDs
matching_comids = centroids_within_target['COMID'].unique().tolist()
# Filter cat GeoDataFrame using matching_comids
filtered_cat = catchments[catchments['COMID'].isin(matching_comids)]
filtered_riv = rivers[rivers['COMID'].isin(matching_comids)]

# Select all outlets [which are flowing into non into the selected basin [filtered_riv]
selected_outlets = filtered_riv.loc[~filtered_riv['NextDownID'].isin(filtered_riv['COMID']), 'COMID'].unique().tolist()

# additional outlets
a = [78000544, 86021891, 86038885, 86023465, 74001129, 86043498, 78000537, 78000626]
selected_outlets = selected_outlets + a
# Replace 74001130 with 74001129
old_value = 74001130
new_value = 74001129
selected_outlets = [new_value if x == old_value else x for x in selected_outlets]

Based on the command above, the sub-basin with a list of `COMID` value that are outlets for Canada and transboundary river basins are extracted.
We may proceed by using the `intersect_topology` to extract whatever located upstream of the sub-basin (selected_outlets) with the list of `COMID` value.

In [ ]:
# `CanTrans` stands for `Canada and Transboundary River basin`:
cantrans_catchments, cantrans_rivers = gm.intersect_topology(
    cat=catchments,
    cat_cols={
        'id':'COMID'
    },
    riv=rivers,
    riv_cols={
        'id':'COMID',
        'next_id':'NextDownID'
    },
    outlet_id=selected_outlets)

Save the extracted subbasin and river network.

In [ ]:
# saving the results into the `output_path` directory
# first, creating the directory
try:
    os.makedirs(output_path)
except FileExistsError:
    pass

# then, saving the data
cantrans_catchments.to_file(os.path.join(output_path, 'merit_cantrans_subbasins.shp'))
cantrans_rivers.to_file(os.path.join(output_path, 'merit_cantrans_rivers.shp'))

## Preparation for subbasin aggregation

Flag all gauging stations subbasin to exclude them from downstream aggregation.

Subbasins that are less than 100km2 in unit area are aggregated towards thier dominant upstream subbasin.

Subbasins that contains gauging outlets are treated differently.

In [6]:
# Read the saved CSV file
# Load the list of subbasin that have gauging stations file
# gauged_subbasin = gpd.read_file('./CanTrans-merit-geofabric/Gauged_MERIT_CanTrans_subbasins.gpkg')
gauged_subbasin = pd.read_csv('./CanTrans-merit-geofabric/all_combined_station_comid.csv')

# Assuming merit_basin is already a DataFrame or GeoDataFrame
# List of required columns
required_cols = ['COMID_WSC','COMID_GAGE','COMID_CAMELS','COMID_CLRHNA']  # Only include the columns you want
# Collect non-NaN values
all_values = []
for col in required_cols:
    non_nan = gauged_subbasin[col].dropna()
    all_values.append(non_nan)
# Concatenate and drop duplicates
combined = pd.concat(all_values, ignore_index=True)
unique_values = combined.drop_duplicates().reset_index(drop=True)
# Optional: convert to DataFrame
unique_df = unique_values.to_frame(name='UniqueValues')
# Add additional point of interest
addition_point_of_interest = [78014244, 72040349, 78014127] # 72055069 < not a station location so removed
new_values_df = pd.DataFrame({'UniqueValues': addition_point_of_interest})
updated_df = pd.concat([unique_df, new_values_df], ignore_index=True)
# Update the geofabric for gauging stations # Add 'Has_gauge' column: 1 otherwise 0 
cantrans_catchments['Has_Gauge'] = cantrans_catchments['COMID'].isin(updated_df['UniqueValues']).astype(int)

### Subbasin aggregation function 

In [ ]:
# Step-Final: # Basin Aggregation Function in Case of Gauge, Lakes and Reservoirs
# Load the required module
import pandas as pd
import geopandas as gpd
import numpy as np

def basin_aggregation(input_basin, input_river, min_SubArea, min_RivSlope, min_RivLength):
    """
    Aggregates basins and rivers based on drainage area, slope, and reservoir masking.

    Parameters:
    - input_basin (GeoDataFrame): Sub-basin geometries with attributes.
    - input_river (GeoDataFrame): River geometries with attributes.
    - min_SubArea (float): Minimum sub-basin area threshold.
    - min_RivSlope (float): Minimum river slope threshold.
    - min_RivLength (float): Minimum river length threshold.

    Returns:
    - agg_basin (GeoDataFrame): Aggregated basins.
    - agg_river (GeoDataFrame): Aggregated rivers.

    # Fix invalid input geometries early but it messup when used
    if not input_basin.is_valid.all():
        input_basin['geometry'] = input_basin.geometry.make_valid()
    if not input_river.is_valid.all():
        input_river['geometry'] = input_river.geometry.make_valid()
    agg_basin = agg_basin[~agg_basin.geometry.is_empty & agg_basin.geometry.notnull()]
    agg_river = agg_river[~agg_river.geometry.is_empty & agg_river.geometry.notnull()]
    
    """
    
    # Check for the minimum slope and length of the river network.
    input_river['slope'] = input_river['slope'].clip(lower=min_RivSlope)
    input_river.loc[input_river['slope'] >= 1.0, 'slope'] = min_RivSlope
    input_river['lengthkm'] = input_river['lengthkm'].clip(lower=min_RivLength)

    # Add the columns from the river network that are not in the basin shapefile
    if 'COMID' not in input_basin.columns:
        raise ValueError("Missing 'COMID' in input_basin.")
        
    # Merge operation to join attribute table from input_river into input_basin
    missing_columns = [col for col in input_river.columns if col not in input_basin.columns[1:]]
    input_basin = input_basin.merge(input_river[missing_columns].copy(), on='COMID', how='left')
    
    # Add flag and variable for future use
    if 'Lake_Cat' not in input_basin.columns:
        input_basin['Lake_Cat'] = 0
    if 'Has_Gauge' not in input_basin.columns:
        input_basin['Has_Gauge'] = 0        
    input_basin['Mask'] = 0
    input_basin.loc[input_basin['NextDownID'] <= 0, 'Mask'] = 1
    input_basin.loc[input_basin['Has_Gauge'] > 0, 'Mask'] = 2
    input_basin.loc[input_basin['Lake_Cat'] > 0, 'Mask'] = 3
    input_basin['agg'] = input_basin['COMID']
    input_basin['aggdown'] = input_basin['NextDownID']

    # Initial filtering
    agg_basin = input_basin[['agg', 'aggdown', 'unitarea', 'uparea', 'Mask']].copy()
    agg_basin = agg_basin[~(((agg_basin['aggdown'] <= 0) & (agg_basin['uparea'] < min_SubArea)) | (agg_basin['Mask'] == 3))]
    lake_subs = input_basin[input_basin['Mask'] == 3]['agg']
    NoSubbasin = len(input_basin)

    while True:
        # Aggregated the headwaters sub-basins
        # Select the Headwater subbasins
        headwaters = (
            ~agg_basin['agg'].isin(agg_basin['aggdown']) &
            (agg_basin['unitarea'] < min_SubArea) &
            (agg_basin['Mask'] < 2)
        )
        small_subbasin = agg_basin[headwaters]
        small_subbasin = small_subbasin[~small_subbasin['aggdown'].isin(lake_subs)].sort_values(by='uparea', ascending=False)
        if not small_subbasin.empty:
            small_subbasin = small_subbasin.rename(columns={'agg': 'aggold', 'aggdown': 'agg'})
            xx = small_subbasin.merge(agg_basin[['agg', 'aggdown']], on='agg', how='left')
            for i in range(len(xx)):
                input_basin.loc[input_basin['agg'] == xx['aggold'].iloc[i], 'aggdown'] = xx['aggdown'].iloc[i]
                input_basin.loc[input_basin['agg'] == xx['aggold'].iloc[i], 'agg'] = xx['agg'].iloc[i]
            agg_basin = input_basin.drop(columns='geometry').groupby(['agg', 'aggdown'], as_index=False).agg({'unitarea': 'sum'})
            agg_basin = agg_basin.rename(columns={'agg': 'COMID', 'aggdown': 'NextDownID'})
            agg_basin = agg_basin.merge(input_basin[['COMID', 'uparea', 'Mask']], on='COMID', how='left')
            agg_basin = agg_basin.rename(columns={'COMID': 'agg', 'NextDownID': 'aggdown'})
            agg_basin = agg_basin[~(((agg_basin['aggdown'] <= 0) & (agg_basin['uparea'] < min_SubArea)) | (agg_basin['Mask'] == 3))]

        # Intermediate Aggregation
        condition = (
            agg_basin['agg'].isin(agg_basin['aggdown']) &
            (agg_basin['unitarea'] < min_SubArea) &
            (agg_basin['Mask'] != 3)
        )
        small_subbasin = agg_basin[condition].sort_values(by='uparea', ascending=False)
        if not small_subbasin.empty:
            for i in range(len(small_subbasin)):
                xx = input_basin[input_basin['COMID'] == small_subbasin['agg'].iloc[i]].index[0]
                if input_basin['unitarea'][input_basin['agg'] == input_basin['agg'].iloc[xx]].sum() < min_SubArea:
                    xy = input_basin[input_basin['NextDownID'] == input_basin['COMID'].iloc[xx]].index
                    if not xy.empty:
                        xz = input_basin['uparea'].iloc[xy].idxmax()
                        if input_basin.loc[xz, 'Mask'] < 2:
                            zz = input_basin[input_basin['aggdown'] == input_basin.loc[xz, 'agg']].index
                            input_basin.loc[input_basin['agg'] == input_basin['agg'].iloc[xz], 'agg'] = input_basin['agg'].iloc[xx]
                            input_basin.loc[input_basin['agg'] == input_basin['agg'].iloc[xz], 'aggdown'] = input_basin['aggdown'].iloc[xx]
                            if not zz.empty:
                                input_basin.loc[zz, 'aggdown'] = input_basin['agg'].iloc[xx]
            agg_basin = input_basin.drop(columns='geometry').groupby(['agg', 'aggdown'], as_index=False).agg({'unitarea': 'sum'})
            agg_basin = agg_basin.rename(columns={'agg': 'COMID', 'aggdown': 'NextDownID'})
            agg_basin = agg_basin.merge(input_basin[['COMID', 'uparea', 'Mask']], on='COMID', how='left')
            agg_basin = agg_basin.rename(columns={'COMID': 'agg', 'NextDownID': 'aggdown'})
            agg_basin = agg_basin[~(((agg_basin['aggdown'] <= 0) & (agg_basin['uparea'] < min_SubArea)) | (agg_basin['Mask'] == 3))]

        # Break if no small sub-basins are left
        if len(agg_basin[agg_basin['unitarea'] < min_SubArea]) == NoSubbasin:
            break
        NoSubbasin = len(agg_basin[agg_basin['unitarea'] < min_SubArea])

    # Final aggregation of the sub-basins
    agg_basin = input_basin.dissolve(by='agg', aggfunc={'unitarea': 'sum'}, as_index=False).rename(columns={'agg': 'COMID'})
    agg_basin = agg_basin.merge(input_basin[['COMID', 'aggdown', 'uparea']].copy(), on='COMID', how='left').rename(columns={'aggdown': 'NextDownID'})

    # Aggregating river network based on the aggregated sub-basins    
    agg_river = input_river.merge(input_basin[['COMID', 'agg']].copy(), on='COMID', how='left')
    agg_river['mask'] = 0
    for i in agg_river['agg'].unique():
        xx = agg_river.index[agg_river['agg'] == i].tolist()
        while True:
            yy = agg_river['uparea'].iloc[xx].idxmax()
            agg_river.at[yy, 'mask'] = 1
            xx = agg_river.index[agg_river['NextDownID'] == agg_river['COMID'].iloc[yy]].tolist()
            if len(xx) < 1:
                break
    agg_river = agg_river[agg_river['mask'] == 1].copy()
    agg_river['slope'] = agg_river['slope'] * agg_river['lengthkm']

    # Final aggregation of the river network
    agg_river = agg_river.dissolve(by='agg', aggfunc={'lengthkm': 'sum', 'slope': 'sum'}, as_index=False).rename(columns={'agg': 'COMID'})

    # Computes the weighted slope per length, checking for zero
    agg_river['slope'] = agg_river['slope'] / agg_river['lengthkm'].replace(0, np.nan)
    agg_river = agg_river.merge(agg_basin[['COMID', 'NextDownID', 'uparea']].copy(), on='COMID', how='left')
    agg_river = agg_river.merge(input_river[['COMID', 'order', 'hillslope']].copy(), on='COMID', how='left')

    # Return the aggregated basin and river shapefile
    return agg_basin, agg_river

Apply the aggregation function

In [ ]:
## Define the level of aggregation required and the minimum length and slope of the river reach.
min_SubArea = 100          # 50, 100, 150
min_RivSlope = 0.0000001   # Minimum accepted value for river slope (WATFLOOD mannua)
min_RivLength = 1.0

# Call for the aggregation function
agg_basin, agg_river = basin_aggregation(cantrans_catchments, cantrans_rivers, min_SubArea, min_RivSlope, min_RivLength)

# Save the aggregated subbasin and river network shapefile
agg_basin['NextDownID'] = agg_basin['NextDownID'].astype('int64')
agg_river['NextDownID'] = agg_river['NextDownID'].astype('int64')
agg_basin = gpd.read_file(os.path.join(output_path, 'agg_all_subbasin', 'agg_MERIT_CanTrans_subbasins.shp'))
agg_river = gpd.read_file(os.path.join(output_path, 'agg_all_subbasin', 'agg_MERIT_CanTrans_rivers.shp'))

Remove all isolated peripheral subbasins that are neither outlets nor gauged to reduce the number of subbasins and model computation burden.

"Eliminate all single, non-gauged peripheral subbasins that do not serve as outlets."

In [17]:
# Read the study domain shapefile and prepare the Outlets
CanTrans = gpd.read_file('./CanTrans-merit-geofabric/CanTrans_MERIT_StudyDomain.shp')
CanTrans = CanTrans.rename(columns={'COMID': 'ID'})
# Explode multipolygons into individual polygon rows
gdf_exploded = CanTrans.explode(index_parts=True, ignore_index=True)

# Find the single largest polygon by area
largest_polygon = gdf_exploded.loc[gdf_exploded.geometry.area.idxmax()]

# Convert it back to a GeoDataFrame (to allow plotting)
largest_gdf = gpd.GeoDataFrame([largest_polygon], columns=gdf_exploded.columns, crs=CanTrans.crs)

# Apply a reasonable inward buffer (e.g., 5 km)
largest_gdf['geometry'] = largest_gdf.geometry.buffer(-2000)

# Optional: Remove invalid or empty geometries created by buffering
largest_gdf = largest_gdf[~largest_gdf['geometry'].is_empty & largest_gdf['geometry'].notnull()]

# Subset the agg_basin and exclude gauged subbasin
exclude_agg_basin = agg_basin[(agg_basin['unitarea'] < 100) & (agg_basin['uparea'] < 100)  & (agg_basin['NextDownID'] < 0)]

# Excluding any subbasin that are gauged subbasin
exclude_agg_basin = exclude_agg_basin.loc[~exclude_agg_basin['COMID'].isin(updated_df['UniqueValues'])]

# Excluding any subbasin that are outlets of gauged subbasin
exclude_agg_basin = exclude_agg_basin.loc[~exclude_agg_basin['COMID'].isin(agg_basin['NextDownID'])]

# Ensure CRS matches
if exclude_agg_basin.crs != largest_gdf.crs:
    largest_gdf = largest_gdf.to_crs(exclude_agg_basin.crs)
    
# Perform spatial join to find intersecting polygons
# 'inner' join keeps only intersecting geometries
intersecting = gpd.sjoin(exclude_agg_basin, largest_gdf, how='inner', predicate='within')

# Filter further for small area that pass the intersection filltering
intersecting = intersecting[~intersecting['COMID'].isin([86012914, 86012955])]

# Add any missing subbasin that need to be kept 
intersecting_comids = pd.concat([
    intersecting['COMID'],
    pd.Series([74003760, 78025467])
], ignore_index=True)

# Subset all comids that needs to be removed 
exclude_agg_comids = exclude_agg_basin.loc[~exclude_agg_basin['COMID'].isin(intersecting_comids), 'COMID']

# Subset: keep only rows where COMID is NOT in intersecting
new_agg_basin = agg_basin[~agg_basin['COMID'].isin(exclude_agg_comids)]
new_agg_river = agg_river[~agg_river['COMID'].isin(exclude_agg_comids)]

# Save the aggregated subbasin and river netwrok shapefile
new_agg_basin.to_file(os.path.join(output_path, 'agg_MERIT_CanTrans_subbasins.shp'))
new_agg_river.to_file(os.path.join(output_path, 'agg_MERIT_CanTrans_rivers.shp'))

Sort the geofabric based on the drainage database rank prepared in advance, in order to reduce computation during the model-agnostic step. This pre-sorting also eliminates the need to re-sort the remapped forcing data from Easymore during the model-specific process.

In [15]:
import os # built-in Python 3.10.2
import geopandas as gpd # version 0.14.3
import xarray as xr
import pandas as pd

# === Configuration ===
output_path = './CanTrans-merit-geofabric/'
agg_basin = gpd.read_file(os.path.join(output_path, 'agg_MERIT_CanTrans_subbasins.shp'))
agg_river = gpd.read_file(os.path.join(output_path, 'agg_MERIT_CanTrans_rivers.shp'))
ddb = xr.open_dataset(os.path.join(output_path, 'MESH_drainage_database.nc'))
rank_column = "Rank"                       # Name of the column to sort by
ddb = xr.open_dataset(os.path.join(output_path, 'MESH_drainage_database.nc'))[['subbasin', 'Rank', 'Next']]

# This flattens the data into tabular form
df_netcdf = ddb.to_dataframe().reset_index()

# --- Merge on subbasin/COMID ---
agg_basin = agg_basin.merge(df_netcdf, left_on='COMID', right_on='subbasin')
agg_river = agg_river.merge(df_netcdf, left_on='COMID', right_on='subbasin')

# === Sort by the Rank column ===
agg_basin_sorted = agg_basin.sort_values(by=rank_column, ascending=True)
agg_river_sorted = agg_river.sort_values(by=rank_column, ascending=True)

# === Save to new shapefile ===
agg_basin_sorted = agg_basin_sorted.drop(columns=['subbasin'])
agg_river_sorted = agg_river_sorted.drop(columns=['subbasin'])

agg_basin_sorted.to_file(os.path.join(output_path, 'sorted_agg_MERIT_CanTrans_subbasins.shp'))
agg_river_sorted.to_file(os.path.join(output_path, 'sorted_agg_MERIT_CanTrans_rivers.shp'))